In [46]:
from sympy import symbols, Rational
from sympy.utilities.codegen import codegen
from sympy.codegen.rewriting import optimize, optims_c99
from sympy.simplify.cse_main import cse
import sympy as sp
from sympy import S
from sympy.printing.c import C99CodePrinter, Assignment
from sympy import init_printing
init_printing()

from codegen.codegen_utils import *
printer = MyPrinter() 

In [47]:
def der_vec(vec,name):
    out = [] 
    n = vec.shape[0]
    for idir in range(3):
        _out = sp.Matrix([0]*n)
        for i in range(n):
            _out[i] = sp.symbols(f"d{name}_dx[{ i + n * idir}]")
        out.append(_out)
    return out

In [48]:
# Setup 
W, dthin, dthick, v2, E, F = symbols("W dthin dthick v2 E F", real=True, nonnegative=True)
Fx, Fy, Fz      = symbols("Fd[0] Fd[1] Fd[2]", real=True)
FX, FY, FZ      = symbols("Fu[0] Fu[1] Fu[2]", real=True)
vX, vY, vZ      = symbols("vu[0] vu[1] vu[2]", real=True)
vx, vy, vz      = symbols("vd[0] vd[1] vd[2]", real=True)
fX, fY, fZ      = symbols("fu[0] fu[1] fu[2]", real=True)
fx, fy, fz      = symbols("fd[0] fd[1] fd[2]", real=True)
HHx,HHy,HHz     = symbols("HHd[0] HHd[1] HHd[2]", real=True)
HHd = sp.Matrix([HHx,HHy,HHz])
Ht = symbols("Ht", real=True)

JJ = symbols("JJ", real=True)
ka, ks, eta     = symbols("ka ks eta", real=True, nonnegative=True)
vdotfh, vdotF   = symbols("vdotfh vdotF", real=True)
z = symbols("z", real=True, nonnegative=True)

alpha = symbols("alp")
tiny = S(1e-20)

Fd = sp.Matrix([Fx,Fy,Fz])
Fu = sp.Matrix([FX,FY,FZ])
vu = sp.Matrix([vX,vY,vZ])
vd = sp.Matrix([vx,vy,vz])
fd = sp.Matrix([fx,fy,fz])
fu = sp.Matrix([fX,fY,fZ])

F2 = F**2

W2 = W**2 
W3 = W2 * W 

gxx,gxy,gxz,gyy,gyz,gzz = symbols("gdd[0] gdd[1] gdd[2] gdd[3] gdd[4] gdd[5]", real=True) 
gdd = sp.Matrix([
    [gxx,gxy,gxz],
    [gxy,gyy,gyz],
    [gxz,gyz,gzz]
])
gXX,gXY,gXZ,gYY,gYZ,gZZ = symbols("guu[0] guu[1] guu[2] guu[3] guu[4] guu[5]", real=True) 
guu = sp.Matrix([
    [gXX,gXY,gXZ],
    [gXY,gYY,gYZ],
    [gXZ,gYZ,gZZ]
])
betaX, betaY, betaZ = symbols("betau[0] betau[1] betau[2]", real=True) 
betau = sp.Matrix([betaX,betaY,betaZ])

betad = gdd * betau

Kxx, Kxy, Kxz, Kyy, Kyz, Kzz = symbols("Kdd[0] Kdd[1] Kdd[2] Kdd[3] Kdd[4] Kdd[5]")
Kdd = sp.Matrix([
    [Kxx,Kxy,Kxz],
    [Kxy,Kyy,Kyz],
    [Kxz,Kyz,Kzz]
])

dgdd_dx = der_symm_tens(gdd,"gdd") 
dbetau_dx = der_vec(betau,"betau")

dalphadx,dalphady,dalphadz = symbols("dalpha[0] dalpha[1] dalpha[2]")
dalpha_dx = sp.Matrix([dalphadx,dalphady,dalphadz])

scalar = ("double", None)
vec = ("double",[3])
fvec = ("double",[4])
symtens = ("double",[6])
tens = ("double",[3,3])

ABI = {
    "W": scalar,
    "dthin": scalar,
    "dthick": scalar,
    "v2": scalar,
    "E": scalar,
    "F": scalar,
    "Fd": vec,
    "Fu": vec,
    "vu": vec,
    "vd": vec,
    "fu": vec,
    "fd": vec,
    "ka": scalar,
    "ks": scalar,
    "eta": scalar,
    "vdotfh": scalar,
    "vdotF": scalar,
    "z": scalar,
    # Metric 
    "alp": scalar,
    "betau": vec,
    "gdd": symtens,
    "guu": symtens,
    "Kdd": symtens,
    # Metric derivatives 
    "dalpha": ("double",[3]),
    "dgdd_dx": ("double",[3*6]),
    "dbetau_dx": ("double",[3*3]),
    # Fluid-frame qties 
    "JJ": scalar,
    "HHd": vec,
    "Ht": scalar,
    "coeffs": ("double",[12])
}

In [49]:
# Closure helpers
B0 = W2 * (E-2*vdotF)
Bthin = W2*E*vdotfh**2
Bthick = (W2-1)/(2*W2+1)*(4*W2*vdotF+(3-2*W2)*E)

an0 = W * B0 + W*(vdotF-E)

av0 = W*B0 

avthin = W*Bthin 
avthick = W*Bthick + W/(2*W2+1)*((2*W2-1)*vdotF+(3-2*W2)*E)

afthin = W*E*vdotfh 
aF0 = -W 
aFthick = W*v2 

anthick = W*Bthick 
anthin = W * Bthin 

helpers = sp.Matrix([B0,Bthin,Bthick,av0,avthin,avthick,afthin,aF0,aFthick,an0,anthin,anthick])

In [50]:
# Closure function 
J = B0 + dthin*Bthin + dthick * Bthick 

# These are the spatial projections, they raise/lower with the spatial metric
Hd = -(av0 + dthin * avthin + dthick*avthick)*vd - dthin * afthin * fd - (aF0 + dthick * aFthick) * Fd 
Hu = -(av0 + dthin * avthin + dthick*avthick)*vu - dthin * afthin * fu - (aF0 + dthick * aFthick) * Fu 

# H2 is H^alpha H_alpha, extra pieces! 
B0_s, Bthin_s, Bthick_s = symbols("coeffs[0] coeffs[1] coeffs[2]",real=True)
av0_s, avthin_s, avthick_s = symbols("coeffs[3] coeffs[4] coeffs[5]",real=True)
afthin_s, aF0_s, aFthick_s = symbols("coeffs[6] coeffs[7] coeffs[8]",real=True)
an0_s, anthin_s, anthick_s = symbols("coeffs[9] coeffs[10] coeffs[11]",real=True)
subs_map = {
    B0: B0_s,
    Bthin: Bthin_s,
    Bthick: Bthick_s,
    av0: av0_s,
    avthin: avthin_s,
    avthick: avthick_s,
    afthin: afthin_s,
    aF0: aF0_s,
    aFthick: aFthick_s,
    an0: an0_s,
    anthin: anthin_s,
    anthick: anthick_s 
}

HH0 = av0**2 * v2 + aF0**2 * F2 + S(2) * av0 * aF0 * vdotF - an0**2 
HHTT = avthick**2 * v2 + aFthick**2 * F2 + 2 * avthick * aFthick * vdotF - anthick**2 
HHtt = avthin**2 * v2 + afthin**2 + 2*avthin*afthin*vdotfh - anthin**2 
HHt = 2 * ( av0*avthin*v2 + aF0*afthin*F + afthin*av0*vdotfh + avthin*aF0*vdotF - anthin*an0)
HHT = 2*( av0 * avthick * v2 + aF0 * aFthick * F2 + aFthick*av0 * vdotF + avthick*aF0 * vdotF - anthick * an0)
HHTt = 2*( avthin * avthick * v2 + afthin*aFthick * F + avthin * aFthick * vdotF + avthick*afthin * vdotfh - anthin * anthick )
H2 = HH0 + dthick * HHT + dthin * HHt + dthick*dthick * HHTT + dthin*dthin*HHtt + dthick*dthin*HHTt 



J_sym = J.subs(subs_map)
Hd_sym = Hd.subs(subs_map)
Hu_sym = Hu.subs(subs_map)
H2_sym = H2.subs(subs_map)

zFunc = (J_sym**2 * z**2 - H2_sym)/E**2

In [51]:
# Pressure tensor 

Puu_thin = E/F2 * Fu * Fu.T 

J_thick = 3/(2*W**2+1)*((2*W**2-1)*E-2*W**2*vdotF)
Hu_thick = Fu/W + - Rational(4,3) * J_thick * W * vu + vu * ( W * vdotF - E * W + J_thick * W )

Puu_thick = Rational(4,3)*J_thick*W**2*vu*vu.T + W * Hu_thick * vu.T + W * vu * Hu_thick.T +  Rational(1,3) * J_thick * guu 

Puu = dthin * Puu_thin + dthick * Puu_thick 

In [52]:
# Jacobian 

# J deriv pieces
JvF = 2*W2*(-1+dthin*E*vdotfh/F + 2*dthick*(W2-1)/(1+2*W2))
JfF = -2*dthin*W2*E*vdotfh**2/F 
# H deriv pieces
HvE = W*W2*(-1-dthin*vdotfh**2 + dthick * (2*W2-3)/(1+2*W2))
HfE = -dthin*W*vdotfh 
HdF = W*(1-dthick*v2-dthin*E*vdotfh/F)
HvvF = 2*W2*W*(1-dthin*E*vdotfh/F-dthick*(v2 + 1/(2*W2*(1+2*W2))))
HffF = 2*dthin*W*E*vdotfh/F
HvfF = 2*dthin*W2*W*E*vdotfh**2/F
HfvF = -dthin*W*E/F

I = sp.Matrix([[1,0,0],[0,1,0],[0,0,1]])

# J derivs 
dJdE = W2 + dthin * vdotfh**2 * W2 + dthick * ((3-2*W2)*(W2-1))/(1+2*W2)
dJdF = JvF * vu + JfF * fu 

# H derivs 
dHdE = HvE * vd + HfE * fd 

dHdF = HdF * I + HvvF * vd * vu.T + HffF * fd * fu.T + HvfF * vd * fu.T + HfvF * fd * vu.T 

J00 = - alpha * W * ( ka + ks - ks * dJdE ) 

J0i = alpha * W * ks * dJdF + alpha * W * (ka+ks) * vu 

Ji0 = -alpha*((ka+ks)*dHdE + W*ka*dJdE*vd)

Jij = -alpha*((ka+ks)*dHdF + W*ka*vd*dJdF.T)

Jacobian = sp.Matrix([
[J00, J0i[0], J0i[1], J0i[2]],
[Ji0[0], Jij[0,0], Jij[0,1], Jij[0,2]],
[Ji0[1], Jij[1,0], Jij[1,1], Jij[1,2]],
[Ji0[2], Jij[2,0], Jij[2,1], Jij[2,2]],
])
Jacobian.is_symmetric()

False

In [53]:
# Source terms 
S0 = alpha * W * (eta + ks * JJ - (ka+ks)*(E-vdotF))
Sd = alpha * W * (eta - ka*JJ)*vd - alpha * (ka+ks)*HHd

sources = sp.Matrix([S0,*Sd])

In [54]:
# Initial guess: transform from lab to fluid in opt. thick regime
Hdotn = Ht/alpha - HHd.dot(betau)/alpha

E_thick = Rational(4,3) * JJ * W2 - 2 * Hdotn * W - Rational(1,3) * JJ
F_thick = Rational(4,3) * JJ * W2 * vd + W * HHd - W * Hdotn *  vd 

In [55]:
# Wavespeeds 
vudir, gammauudir, betaudir, Fudir = symbols("vDIR gammaDD betaDIR FDIR", real=True)
ABI["vDIR"] = scalar
ABI["gammaDD"] = scalar
ABI["betaDIR"] = scalar
ABI["FDIR"] = scalar

pp = alpha * vudir/W 
r = sp.sqrt(alpha**2*gammauudir*(2*W**2+1)-2*W**2*pp**2)

lmthin = -betaudir - alpha * sp.Abs(Fudir)/F 
lmthick = sp.Min(-betaudir + (2*pp*W**2-r)/(2*W**2+1), -betaudir+pp)

lpthin = -betaudir + alpha * sp.Abs(Fudir)/F 
lpthick =  sp.Max(-betaudir + (2*pp*W**2+r)/(2*W**2+1), -betaudir+pp)

lm = dthin * lmthin + dthick * lmthick 
lp = dthin * lpthin + dthick * lpthick 

In [56]:
PXX, PXY, PXZ, PYY, PYZ, PZZ = symbols("Puu[0][0] Puu[0][1] Puu[0][2] Puu[1][1] Puu[1][2] Puu[2][2]")
_PUU = sp.Matrix(
    [[PXX,PXY,PXZ],
     [PXY,PYY,PYZ],
     [PXZ,PYZ,PZZ]]
)
ABI["Puu"] = ("double",[3,3])
PUD = _PUU * gdd 

Esource = 0 
for i in range(3):
    Esource -= Fu[i] * dalpha_dx[i] 
    for j in range(3):
            Esource += alpha * _PUU[i,j] * Kdd[i,j]

Fsource = sp.MutableDenseMatrix([0]* (3))

for i in range(3):
    Fsource[i] -= E * dalpha_dx[i]
    for j in range(3):
        Fsource[i] += Fd[j] * dbetau_dx[i][j]
        for k in range(3):
            Fsource[i] += alpha/S(2) * _PUU[j,k] * dgdd_dx[i][j,k]

In [57]:
flist = [] 

outputs = ["helpers"]
out_abi = {"helpers": ("double",[12])}
flist.append(
    make_function(helpers,printer,"m1_closure_helpers",ABI,outputs,out_abi,layout="flat")
)

outputs = ["out"]
out_abi = {"out": ("double", None)}
flist.append(
    make_function([zFunc],printer,"m1_z_rootfind",ABI,outputs,out_abi,layout="flat")
)

outputs = ["J", "Hd"]
out_abi = {"J": scalar, "Hd": vec}
flist.append(
    make_function([J_sym,Hd_sym],printer,"m1_J_Hd",ABI,outputs,out_abi,layout="flat")
)

outputs = ["Puu"]
out_abi = {"Puu": ("double",[3,3])}
flist.append(
    make_function([Puu],printer,"m1_PUU",ABI,outputs,out_abi,layout="expand")
)

outputs = ["dE", "dF"]
out_abi = {"dE": scalar, "dF": vec}
flist.append(
    make_function([S0,Sd],printer,"m1_source",ABI,outputs,out_abi,layout="expand")
)

outputs = ["J"]
out_abi = {"J": ("double",[4,4])}
flist.append(
    make_function([Jacobian],printer,"m1_jacobian",ABI,outputs,out_abi,layout="expand")
)

outputs = ["Et", "Fdt"]
out_abi = {"Et": scalar, "Fdt": vec}
flist.append(
    make_function([E_thick,F_thick],printer,"m1_fluid_to_lab_thick",ABI,outputs,out_abi,layout="expand")
)


outputs = ["cm", "cp"]
out_abi = {"cm": scalar, "cp": scalar}
flist.append(
    make_function([lm,lp],printer,"m1_wavespeeds",ABI,outputs,out_abi,layout="expand")
)

outputs = ["dE", "dF"]
out_abi = {"dE": scalar, "dF": vec}
flist.append(
    make_function([Esource,Fsource],printer,"m1_geom_source_terms",ABI,outputs,out_abi,layout="expand")
)

printed_functions = '\n' + '\n'.join(flist)

In [58]:
file = '''
/****************************************************************************/
/*                        M1 helpers, SymPy generated                       */
/****************************************************************************/
#ifndef GRACE_M1_SUBEXPR_HH
#define GRACE_M1_SUBEXPR_HH

#include <Kokkos_Core.hpp>\n
''' + printed_functions + '''
#endif 
'''
with open("m1_subexpressions.hh","w") as ff:
    ff.write(file)